In [1]:
#!pip uninstall huetracer -y
#!pip install huetracer
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
import numpy as np
import pandas as pd
import huetracer
import scvi
import gc
import math
import bin2cell as b2c
import torch
from itertools import cycle
import warnings
import importlib
import sys

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"
pd.set_option('display.max_columns', None)
sc.set_figure_params(figsize=[10,10],dpi=100)

scvi.settings.seed = 0
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else"mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
device_str = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
warnings.filterwarnings("ignore", message=".*must be within the support of the distribution.*")

Seed set to 0


Using device: mps


In [6]:
### parameters to be input
#SAMPLE_NAME = 'G1'
SAMPLE_NAME = 'FFPE9'
lib_id = SAMPLE_NAME # list(sp_adata.uns['spatial'].keys())[0]

path = os.path.expanduser("~")+"/Desktop/space/GCTB_bin2cell"
save_path_for_today = os.path.expanduser("~")+"/tmp/outputs/250610_" + SAMPLE_NAME
visium_path = "/Volumes/Public/data/giant_cell_tumor_sue/visium_HD/"
source_image_path = visium_path + "image/" + SAMPLE_NAME + ".tif"
expression_path = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_002um"
expression_path_8um = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_008um"
###

# area to be analyzed
# ## GCTB spatial G1, FFPE9
mask_large_x1, mask_large_x2, mask_large_y1, mask_large_y2 = 450, 1950, 250, 1750

Species = "Human"
# Species = "Mouse"

# List of target gene names
if Species == "Human":
target_genes = [
    'TNFSF11', 'LIF', 'TNFRSF11A', 'LIFR',
    'CTSK', 'CTSB', 'LMNB1', 'MEGF10',
    'MMP9', 'MMP13', 'SPP1', 'GRN',
    'NOTCH3', 'SDC2', 'TIMP1', 'SPARC',
    'ACP5'
    # Add more genes here if needed
]
    prefix_mt = 'MT-'
else:
    target_genes = [
        'Col1a1' # Add more genes here if needed
    ]
    prefix_mt = 'mt-'

annotation_dict = {
    'C0':  'Tumor',
    'C1':  'Tumor',
    'C2':  'Tumor',
    'C3':  'Tumor',
    'C4':  'Tumor',
    'C5':  'Tumor',
    'C6':  'Tumor',
    'C7':  'Tumor',
    'C8':  'Tumor',
    'C9':  'Tumor',
    'C10': 'Tumor',
    'C11': 'Tumor',
    'C12': 'Tumor',
    'C13': 'Tumor',
    'C14': 'Tumor',
    'C15': 'Tumor',
    'C16': 'Tumor',
    'C17': 'Tumor',
    'C18': 'Tumor',
    'C19': 'Tumor',
    'C20': 'Monocyte',
    'C21': 'Monocyte',
    'C22': 'Tumor',
    'C23': 'Tumor',
    'C24': 'Doublet',
    'C25': 'Doublet',
    'C26': 'Lymphocyte',
    'C27': 'Doublet',
    'C28': 'Lymphocyte',
    'C29': 'Doublet',
    'C30': 'Doublet',
    'C31': 'Doublet',
    'C32': 'Monocyte',
    'C33': 'Monocyte',
    'C34': 'Monocyte',
    'C35': 'Tumor',
    'C36': 'Tumor',
    'C37': 'Monocyte',
    'C38': 'Monocyte',
    'C39': 'Monocyte',
    'C40': 'Monocyte',
    'C41': 'Tumor',
    'C42': 'Tumor',
    'C43': 'Tumor',
    'C44': 'GiantCell',
    'C45': 'Doublet',
    'C46': 'GC_proliferating',
    'C47': 'Tumor',
    'C48': 'Doublet',
    'C49': 'CAFibroblast',
    'C50': 'Tumor',
    'C51': 'GiantCell',
    'C52': 'GiantCell',
    'C53': 'Doublet',
    'C54': 'Endothelial',
    'C55': 'Endothelial',
    'C56': 'Doublet',
    'C57': 'Monocyte',
    'C58': 'Doublet',
    'C59': 'CAFibroblast',
    'C60': 'Tumor',
    'C61': 'Doublet',
    'C62': 'Monocyte',
    'C63': 'Endothelial',
    'C64': 'Doublet',
    'C65': 'Tumor',
    'C66': 'Tumor',
    'C67': 'Monocyte',
    'C68': 'GiantCell',
    'C69': 'Doublet',
}

# setting for filenames
label_image_filename = "he_labels_image.pdf"
h5ad_filename = SAMPLE_NAME + "_b2c.h5ad"
h5ad_full_filename = SAMPLE_NAME + "_2um.h5ad"
h5ad_predicted_full_filename = SAMPLE_NAME + "_nucleus_predicted.h5ad"
h5ad_sc_filtered_full_filename = SAMPLE_NAME + "_single_cell_filtered.h5ad"
h5ad_sc_microenvironment_full_filename = SAMPLE_NAME + "_single_cell_microenvironment.h5ad"
save_spatial_plot_path = os.path.join(save_path_for_today, "cropped_spatial_plot.svg")
save_svg_path = os.path.join(save_path_for_today, "spatial_salvage_labels.svg")
h5ad_save_path = os.path.join(save_path_for_today, h5ad_filename)
h5ad_full_save_path = os.path.join(save_path_for_today, h5ad_full_filename)
h5ad_predicted_full_save_path = os.path.join(save_path_for_today, h5ad_predicted_full_filename)
h5ad_sc_filtered_full_save_path = os.path.join(save_path_for_today, h5ad_sc_filtered_full_filename)
h5ad_microenvironment_full_save_path = os.path.join(save_path_for_today, h5ad_sc_microenvironment_full_filename)

os.chdir(path)
os.makedirs(save_path_for_today, exist_ok=True)


# Label transfer from single cell transcriptome annotation to spacial dataset

In [ ]:
# Spatial data load and curation
sp_adata_raw = sc.read_h5ad(h5ad_save_path)
cell_mask = ((sp_adata_raw.obs['array_row'] >= mask_large_x1) & 
             (sp_adata_raw.obs['array_row'] <= mask_large_x2) & 
             (sp_adata_raw.obs['array_col'] >= mask_large_y1) & 
             (sp_adata_raw.obs['array_col'] <= mask_large_y2)
            )
sp_adata = sp_adata_raw[cell_mask].copy()
sp_adata.var_names_make_unique()
sp_adata = sp_adata[sp_adata.obs['bin_count']>5].copy() # min 6 bins
#need integers for seurat v3 hvgs
sp_adata.X = np.round(sp_adata.X).copy()
sp_adata.raw = sp_adata.copy()
sp_adata.var['MT'] = sp_adata.var_names.str.startswith(prefix_mt)
sc.pp.calculate_qc_metrics(sp_adata, qc_vars=['MT'], percent_top=None, log1p=False, inplace=True)
sc.pl.highest_expr_genes(sp_adata, n_top=30)
sc.pl.violin(sp_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_MT'], jitter=0.4, multi_panel=True)
sc.pl.scatter(sp_adata, 'total_counts', 'n_genes_by_counts', color='pct_counts_MT', size=40)
sc.pl.scatter(sp_adata, x='total_counts', y='pct_counts_MT')
fig = plt.figure()
sns.displot(sp_adata.obs['pct_counts_MT'][sp_adata.obs['pct_counts_MT'] < 10], kde=False)
plt.show()

print('Total number of cells: {:d}'.format(sp_adata.n_obs))

# Filtering
# sc.pp.filter_cells(sp_adata, min_genes=40)
# sc.pp.filter_genes(sp_adata, min_cells=100)
# print('Number of cells after low-quality cell filter: {:d}'.format(sp_adata.n_obs))

sc.pp.filter_cells(sp_adata, min_counts = 200)
#sc.pp.filter_cells(sp_adata, max_counts = 2000)
print('Number of cells after count filter: {:d}'.format(sp_adata.n_obs))

sp_adata = sp_adata[sp_adata.obs['pct_counts_MT'] < 10]
print('Number of cells after MT filter: {:d}'.format(sp_adata.n_obs))

sc.pl.highest_expr_genes(sp_adata, n_top=30)
sc.pl.violin(sp_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_MT'], jitter=0.4, multi_panel=True)
sc.pl.scatter(sp_adata, 'total_counts', 'n_genes_by_counts', color='pct_counts_MT', size=40)
sc.pl.scatter(sp_adata, x='total_counts', y='pct_counts_MT')
fig = plt.figure()
sns.displot(sp_adata.obs['pct_counts_MT'][sp_adata.obs['pct_counts_MT'] < 10], kde=False)
plt.show()
# save raw count data
sp_adata.layers["counts"] = sp_adata.X.copy()
sp_adata.raw = sp_adata

# reset with raw count data
sp_adata_sponly = sp_adata.copy()
sp_adata_sponly.X = sp_adata_sponly.raw.X.copy()
sp_adata_sponly.var = sp_adata_sponly.raw.var.copy()
sp_adata_sponly.layers["counts"] = sp_adata_sponly.X.copy()
sc.pp.normalize_total(sp_adata_sponly)
sc.pp.log1p(sp_adata_sponly)
sc.pp.highly_variable_genes(sp_adata_sponly,
                            flavor = 'seurat_v3',
                            n_top_genes=2000,
                            layer = "counts",
                            subset = False)
sc.tl.pca(sp_adata_sponly, svd_solver='arpack',mask_var='highly_variable', n_comps=20)
sc.pp.neighbors(sp_adata_sponly, random_state=SEED)
sc.tl.umap(sp_adata_sponly, random_state=SEED)
sc.tl.leiden(sp_adata_sponly, resolution=1, flavor="igraph", n_iterations=2, key_added='leiden', random_state=SEED)
sc.pl.pca(sp_adata_sponly, color='total_counts', components=['1,2', '2,3', '1,3'])
sc.pl.pca_variance_ratio(sp_adata_sponly, log=True)
sc.pl.umap(sp_adata_sponly, color=['leiden', 'total_counts'],use_raw=False)
sc.pl.spatial(sp_adata_sponly, color='leiden',
              title='Annotation for nuclei, sp data only', size=20, img_key='hires', legend_fontsize=5,
              spot_size=1, frameon=False, )
sp_adata.obs['leiden_nucleus'] = sp_adata_sponly.obs['leiden']
sp_adata.obsm['X_PCA_nucleus'] = sp_adata_sponly.obsm['X_pca']
sp_adata.obsm['X_umap_nucleus'] = sp_adata_sponly.obsm['X_umap']

In [4]:
# # Spatial data of 8um bin, option for comparing
# sp_adata_bin = b2c.read_visium(expression_path_8um, source_image_path = source_image_path)
# sp_adata_bin.var_names_make_unique()
# cell_mask = ((sp_adata_bin.obs['array_row'] / sp_adata_bin.uns['spatial'][lib_id]['scalefactors']['microns_per_pixel'] >= mask_large_x1) & 
#              (sp_adata_bin.obs['array_row'] / sp_adata_bin.uns['spatial'][lib_id]['scalefactors']['microns_per_pixel'] <= mask_large_x2) & 
#              (sp_adata_bin.obs['array_col'] / sp_adata_bin.uns['spatial'][lib_id]['scalefactors']['microns_per_pixel'] >= mask_large_y1) & 
#              (sp_adata_bin.obs['array_col'] / sp_adata_bin.uns['spatial'][lib_id]['scalefactors']['microns_per_pixel'] <= mask_large_y2)
#             )
# sp_adata_bin = sp_adata_bin[cell_mask]
# sp_adata_bin.var_names_make_unique()
# sp_adata_bin.raw = sp_adata_bin.copy()
# sp_adata_bin.var['MT'] = sp_adata_bin.var_names.str.startswith('MT-')
# sc.pp.calculate_qc_metrics(sp_adata_bin, qc_vars=['MT'], percent_top=None, log1p=False, inplace=True)
# # sc.pl.highest_expr_genes(sp_adata_bin, n_top=30)
# # sc.pl.violin(sp_adata_bin, ['n_genes_by_counts', 'total_counts', 'pct_counts_MT'], jitter=0.4, multi_panel=True)
# # sc.pl.scatter(sp_adata_bin, 'total_counts', 'n_genes_by_counts', color='pct_counts_MT', size=40)
# # sc.pl.scatter(sp_adata_bin, x='total_counts', y='pct_counts_MT')
# # fig = plt.figure()
# # sns.displot(sp_adata_bin.obs['pct_counts_MT'][sp_adata_bin.obs['pct_counts_MT'] < 10], kde=False)
# # plt.show()

# print('Total number of cells: {:d}'.format(sp_adata_bin.n_obs))
# # Filtering
# sc.pp.filter_cells(sp_adata_bin, min_genes=40)
# sc.pp.filter_genes(sp_adata_bin, min_cells=100)
# print('Number of cells after low-quality cell filter: {:d}'.format(sp_adata_bin.n_obs))

# sc.pp.filter_cells(sp_adata_bin, min_counts = 100)
# sc.pp.filter_cells(sp_adata_bin, max_counts = 1500)
# print('Number of cells after count filter: {:d}'.format(sp_adata_bin.n_obs))

# sp_adata_bin = sp_adata_bin[sp_adata_bin.obs['pct_counts_MT'] < 10]
# print('Number of cells after MT filter: {:d}'.format(sp_adata_bin.n_obs))

# # sc.pl.highest_expr_genes(sp_adata_bin, n_top=30)
# # sc.pl.violin(sp_adata_bin, ['n_genes_by_counts', 'total_counts', 'pct_counts_MT'], jitter=0.4, multi_panel=True)
# # sc.pl.scatter(sp_adata_bin, 'total_counts', 'n_genes_by_counts', color='pct_counts_MT', size=40)
# # sc.pl.scatter(sp_adata_bin, x='total_counts', y='pct_counts_MT')
# # fig = plt.figure()
# # sns.displot(sp_adata_bin.obs['pct_counts_MT'][sp_adata_bin.obs['pct_counts_MT'] < 10], kde=False)
# # plt.show()
# # save raw count data
# sp_adata_bin.layers["counts"] = sp_adata_bin.X.copy()
# sp_adata_bin.raw = sp_adata_bin

# # reset with raw count data
# sp_adata_sponly = sp_adata_bin.copy()
# sp_adata_sponly.X = sp_adata_bin.raw.X.copy()
# sp_adata_sponly.var = sp_adata_bin.raw.var.copy()
# sp_adata_sponly.layers["counts"] = sp_adata_sponly.X.copy()
# sc.pp.normalize_total(sp_adata_sponly)
# sc.pp.log1p(sp_adata_sponly)
# sc.pp.highly_variable_genes(sp_adata_sponly,
#                             flavor = 'seurat_v3',
#                             n_top_genes=2000,
#                             layer = "counts",
#                             subset = False)
# sc.tl.pca(sp_adata_sponly, svd_solver='arpack',mask_var='highly_variable', n_comps=20)
# sc.pp.neighbors(sp_adata_sponly, random_state=SEED)
# sc.tl.umap(sp_adata_sponly, random_state=SEED)
# sc.tl.leiden(sp_adata_sponly, resolution=0.3, key_added='leiden', random_state=SEED)
# sc.pl.pca(sp_adata_sponly, color='total_counts', components=['1,2', '2,3', '1,3'])
# sc.pl.pca_variance_ratio(sp_adata_sponly, log=True)
# #sc.tl.rank_genes_groups(sp_adata_sponly, 'leiden', method='wilcoxon', use_raw=False)
# #sc.pl.rank_genes_groups(sp_adata_sponly, n_genes=25, sharey=False)
# #sp_adata_sponly.var.sort_values("variances_norm",ascending=False).head(10)
# sc.pl.umap(sp_adata_sponly, color=['leiden', 'total_counts'],use_raw=False)
# sc.pl.umap(sp_adata_sponly, color=['RUNX2', 'CTSK', 'LYZ'],use_raw=False)
# # sc.pl.umap(sp_adata_sponly, color=['CD163', 'COL12A1', 'COL1A1'],use_raw=False)
# # sc.pl.umap(sp_adata_sponly, color=['IL2RG', 'CDH5', 'TRAC'],use_raw=False)
# sc.pl.spatial(sp_adata_sponly, color='leiden',
#               title='Annotation for 8um-bin', size=20, img_key='hires', legend_fontsize=5,
#               spot_size=1, frameon=False, )

In [ ]:
# SCVI label transfer
sp_adata_predicted = huetracer.run_scvi_label_transfer(
    filtered_sc_adata=filtered_sc_adata,
    sp_adata=sp_adata,
    annotation_dict=annotation_dict,
    lib_id=lib_id,
    sample_name=SAMPLE_NAME,
    save_path_for_today=save_path_for_today,
    h5ad_predicted_full_save_path=h5ad_predicted_full_save_path,
    device=device,
    num_workers="auto",
    batch_size = 2048,
    max_epochs = 100
)